# Cubit Mesh Export Showcase

This result-saved notebook is the presentation layer for `cubit-mesh-export`: the Coreform Cubit plugin that turns labeled Cubit meshes into solver-facing artifacts, especially Netgen `.vol` for NGSolve/Radia.

The practical message for the talk is simple:

1. **Cubit owns the meshing and labels.** Blocks and sidesets become materials, boundaries, and force-integration surfaces.
2. **`.vol` is the computation route.** It preserves labels and high-order curved nodes for NGSolve.
3. **GMSH / Nastran are export routes, not the primary Radia solve route.** They are useful for visualization and downstream solver interchange.
4. **Validation is machine-readable.** Every public claim below is backed by adjacent `validation_*_summary.json` payloads and this notebook's own result sidecar.

In [1]:
from pathlib import Path
import json
from IPython.display import Markdown, display

candidates = [
    Path('cubit_mesh_export_showcase_results.json'),
    Path('docs/cubit_mesh_export/cubit_mesh_export_showcase_results.json'),
]
result_path = next((p for p in candidates if p.exists()), None)
assert result_path is not None, candidates
data = json.loads(result_path.read_text(encoding='utf-8'))
summary = data['summary']
print('result_json:', result_path)
for key in ['validation_script_count', 'validation_summary_count', 'python_script_count', 'asset_count']:
    print(f'{key}: {summary[key]}')
print('high_order_hex_max_abs_error_percent:', summary['hex_high_order_max_abs_error_percent'])
print('poisson_error_drop_order1_to_3:', summary['hex_poisson_error_drop_ratio_order1_to_3'])
assert summary['validation_script_count'] == 18
assert summary['validation_summary_count'] == 18
assert summary['all_validation_summaries_present'] is True

result_json: cubit_mesh_export_showcase_results.json
validation_script_count: 18
validation_summary_count: 18
python_script_count: 25
asset_count: 15
high_order_hex_max_abs_error_percent: 0.211
poisson_error_drop_order1_to_3: 20.37433155080214


## What Gets Validated

The validation corpus is deliberately first-order and readable. It checks the surfaces that later high-order or production workflows depend on: boundary orientation, pressure and traction resultants, material-interface incidence, tetrahedron/surface quality, P1 trace matrices, and FEM/BEM topology. This is the level where broken labels or flipped normals are cheapest to catch.

In [2]:
rows = []
for rec in data['validation_records']:
    payload = rec.get('summary') or {}
    kind = payload.get('schema') or payload.get('kind') or rec['name'].replace('.py', '')
    checks = payload.get('checks') if isinstance(payload, dict) else None
    if isinstance(checks, dict):
        check_preview = ', '.join(list(checks.keys())[:4])
    else:
        check_preview = ', '.join(rec.get('summary_keys', [])[:4])
    rows.append((rec['path'], kind, rec['line_count'], check_preview))

header = '| Script | Summary kind | Lines | Representative keys |\n|---|---|---:|---|'
body = [f'| `{path}` | `{kind}` | {lines} | {keys} |' for path, kind, lines, keys in rows]
display(Markdown('\n'.join([header] + body)))

| Script | Summary kind | Lines | Representative keys |
|---|---|---:|---|
| `validation_test/cubit_mesh_export/validation_p1_surface_triangle_element_summary.py` | `p1_surface_triangle_element_summary_validation` | 113 | area, constant_load_integral, gradient_partition_residual, mass_integral_of_one |
| `validation_test/cubit_mesh_export/validation_vol_boundary_condition_assignment.py` | `vol_boundary_condition_assignment_validation` | 189 | clean_boundary_count, clean_condition_counts, clean_ok, missing_boundary_count |
| `validation_test/cubit_mesh_export/validation_vol_boundary_edge_inventory.py` | `vol_boundary_edge_inventory_validation` | 174 | boundary_count, overused_edges_total, perimeter_edges_total, shared_diagonal_edges_total |
| `validation_test/cubit_mesh_export/validation_vol_boundary_inventory.py` | `validation_vol_boundary_inventory` | 215 | builtin_all_faces_have_four_trace_nodes, builtin_all_faces_have_two_triangles, builtin_boundary_count, builtin_boundary_names |
| `validation_test/cubit_mesh_export/validation_vol_boundary_normal_vectors.py` | `netgen_vol_boundary_normal_vectors_validation` | 225 | builtin_boundary_names, builtin_vector_area_errors, builtin_max_vector_area_error, builtin_max_planar_ratio_error |
| `validation_test/cubit_mesh_export/validation_vol_boundary_oriented_edges.py` | `vol_boundary_oriented_edges_validation` | 136 | first_triangle_oriented_edges, first_triangle_signs, hcurl_edge_ids, is_closed_manifold |
| `validation_test/cubit_mesh_export/validation_vol_boundary_pressure_force.py` | `netgen_vol_boundary_pressure_force_validation` | 133 | builtin_uniform_total_force_norm_N, builtin_zmax_force_N, builtin_zmax_force_abs_error_N |
| `validation_test/cubit_mesh_export/validation_vol_boundary_pressure_moment.py` | `netgen_vol_boundary_pressure_moment_validation` | 189 | builtin_uniform_total_force_norm_N, builtin_uniform_total_moment_norm_Nm, builtin_zmax_centroid_m, builtin_zmax_force_N |
| `validation_test/cubit_mesh_export/validation_vol_boundary_pressure_resultant.py` | `netgen_vol_boundary_pressure_resultant_validation` | 144 | builtin_uniform_total_force_norm_N, builtin_uniform_total_moment_norm_Nm, builtin_uniform_force_balance_ratio, builtin_uniform_moment_balance_ratio |
| `validation_test/cubit_mesh_export/validation_vol_boundary_tet_face_incidence.py` | `vol_boundary_tet_face_incidence_validation` | 158 | domain_material_mismatch_count, exterior_row_count, exterior_surface_triangles, interface_adjacent_material_numbers |
| `validation_test/cubit_mesh_export/validation_vol_boundary_traction_moment.py` | `netgen_vol_boundary_traction_moment_validation` | 179 | builtin_zmax_centroid_m, builtin_zmax_force_N, builtin_zmax_moment_Nm, builtin_zmax_shifted_moment_Nm |
| `validation_test/cubit_mesh_export/validation_vol_fem_bem_topology.py` | `validation_vol_fem_bem_topology` | 218 | builtin_closed_euler_characteristic, builtin_closed_hcurl_edges, builtin_closed_rwg_dof_edges, builtin_closed_surface_edges |
| `validation_test/cubit_mesh_export/validation_vol_material_interface.py` | `netgen_vol_material_interface_inventory_validation` | 222 | built_in, kind, validation_class |
| `validation_test/cubit_mesh_export/validation_vol_mesh_health.py` | `netgen_vol_mesh_health_validation` | 201 | clean_status, clean_ok, sliver_status, sliver_tet_quality_ok |
| `validation_test/cubit_mesh_export/validation_vol_p1_trace_matrix.py` | `vol_p1_fem_bem_trace_matrix_validation` | 143 | cols, interior_node_ids, matrix_shape, n_trace_nodes |
| `validation_test/cubit_mesh_export/validation_vol_surface_closure.py` | `validation_vol_surface_closure` | 154 | clean_abs_volume_rel_error, clean_boundary_orientation, clean_surface_area, clean_vector_area_norm_over_area |
| `validation_test/cubit_mesh_export/validation_vol_surface_triangle_quality.py` | `netgen_vol_surface_triangle_quality_validation` | 207 | unit_min_radius_ratio_quality, unit_expected_right_face_quality, unit_max_radius_ratio_quality, unit_max_edge_ratio |
| `validation_test/cubit_mesh_export/validation_vol_tet_quality.py` | `netgen_vol_tetrahedron_quality_validation` | 198 | right_volume, right_surface_area, right_inradius, right_circumradius |

## The Boundary Force Story

For a Cubit sideset to be useful in electromagnetics, it must be more than a name. It needs oriented triangles, outward vector areas, and stable force/moment reduction. The pressure and traction validations below are the small closed-form checks behind Maxwell-stress and acoustic/electromagnetic load transfer.

In [3]:
def find_summary(name):
    for rec in data['validation_records']:
        if rec['name'] == name:
            return rec['summary']
    raise KeyError(name)

normal = find_summary('validation_vol_boundary_normal_vectors.py')
pressure = find_summary('validation_vol_boundary_pressure_resultant.py')
traction = find_summary('validation_vol_boundary_traction_moment.py')
health = find_summary('validation_vol_mesh_health.py')

print('[boundary normals]')
for row in normal.get('builtin_boundary_rows', normal.get('rows', []))[:6]:
    name = row.get('boundary') or row.get('name')
    area = row.get('area') or row.get('surface_area')
    vector = row.get('vector_area') or row.get('area_vector')
    print(f'  {name}: area={area}, vector={vector}')

print('\n[pressure resultant checks]')
for key, value in (pressure.get('checks') or pressure).items():
    if 'force' in key or 'moment' in key or 'error' in key:
        print(f'  {key}: {value}')

print('\n[traction moment checks]')
for key, value in (traction.get('checks') or traction).items():
    if 'force' in key or 'moment' in key or 'error' in key:
        print(f'  {key}: {value}')

print('\n[mesh health statuses]')
for row in health.get('rows', []):
    print(f"  {row.get('name')}: status={row.get('status')} ok={row.get('ok')}")

[boundary normals]

[pressure resultant checks]
  builtin_uniform_total_force_norm_N: 0.0
  builtin_uniform_total_moment_norm_Nm: 0.0
  builtin_uniform_force_balance_ratio: 0.0
  builtin_uniform_moment_balance_ratio: 0.0
  builtin_uniform_absolute_force_sum_N: 124.0
  builtin_zmax_force_N: [0.0, 0.0, 12.0]
  builtin_zmax_moment_Nm: [18.0, -12.0, 0.0]
  builtin_zmax_shifted_moment_Nm: [0.0, 4.440892098500626e-16, 0.0]

[traction moment checks]
  builtin_zmax_force_N: [6.0, -12.0, 18.0]
  builtin_zmax_moment_Nm: [87.0, 12.0, -21.0]
  builtin_zmax_shifted_moment_Nm: [0.0, 1.3322676295501878e-15, 4.440892098500626e-16]
  builtin_zmax_resultant_force_error_N: 0.0
  builtin_zmax_resultant_moment_error_Nm: 0.0

[mesh health statuses]


## High-Order Hexahedral Meshes In NGSolve

This is the headline demo. Netgen can curve tetrahedra from CAD, but the Cubit export path can preserve **curved high-order hexahedra** in a `.vol` file. NGSolve loads those nodes directly. The important gotcha is also shown by the example source: load the `.vol` as-is and do **not** call `mesh.Curve()` on an already curved file.

In [4]:
volume_rows = data['hex_highorder']['volume_rows']
poisson_rows = data['hex_highorder']['poisson_rows']

header = '| Order | HEX count | Sphere volume error | Poisson ndof | Poisson L2 error |\n|---:|---:|---:|---:|---:|'
body = []
for v, p in zip(volume_rows, poisson_rows):
    body.append(
        f"| {v['order']} | {p['hex']} | {v['error_percent']:+.3f}% | {p['ndof']} | {p['l2_error']:.3e} |"
    )
display(Markdown('\n'.join([header] + body)))

print(data['hex_highorder']['volume_stdout'])
print(data['hex_highorder']['poisson_stdout'])
assert max(abs(r['error_percent']) for r in volume_rows if r['order'] >= 2) < 1.0
assert poisson_rows[2]['l2_error'] < poisson_rows[1]['l2_error'] < poisson_rows[0]['l2_error']

| Order | HEX count | Sphere volume error | Poisson ndof | Poisson L2 error |
|---:|---:|---:|---:|---:|
| 1 | 56 | -23.359% | 79 | 8.001e-06 |
| 2 | 56 | -0.211% | 517 | 1.267e-06 |
| 3 | 56 | +0.131% | 1651 | 3.927e-07 |

Curved-hex sphere in NGSolve  (R=0.05 m, V_exact=5.235988e-04 m^3)
  order 1  {'HEX': 56}  V=4.012929e-04  err=-23.359%
  order 2  {'HEX': 56}  V=5.224933e-04  err=-0.211%
  order 3  {'HEX': 56}  V=5.242872e-04  err=+0.131%
PASS: curved high-order hex converges to the analytic sphere (order>=2 < 1%)

Poisson on the curved hex sphere (p-refinement, fixed 56-hex mesh):
  order 1  HEX=56  ndof=79  ||u_h - u_exact||_L2 = 8.001e-06
  order 2  HEX=56  ndof=517  ||u_h - u_exact||_L2 = 1.267e-06
  order 3  HEX=56  ndof=1651  ||u_h - u_exact||_L2 = 3.927e-07
PASS: FE error falls with order on the curved hex mesh (8.00e-06 -> 3.93e-07) -- NGSolve solves on high-order Cubit hexes



## Source Excerpts To Show During Q&A

The notebook carries source hashes for all 25 Python scripts. The excerpts below are the two code paths most likely to come up in questions: how the curved hex `.vol` is consumed, and how boundary vector areas are reduced into named sideset rows.

In [5]:
source_by_path = {rec['path']: rec.get('source') for rec in data['validation_records']}

def read_repo_relative_text(path):
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / path
        if candidate.exists():
            return candidate.read_text(encoding='utf-8')
    raise FileNotFoundError(path)

# Non-validation showcase helpers are maintained as live docs files, not in a source archive.
for path in [
    'docs/cubit_mesh_export/hex_sphere_highorder/hex_sphere_curved_ngsolve.py',
]:
    source_by_path.setdefault(path, read_repo_relative_text(path))

for path in [
    'docs/cubit_mesh_export/hex_sphere_highorder/hex_sphere_curved_ngsolve.py',
    'validation_test/cubit_mesh_export/validation_vol_boundary_normal_vectors.py',
]:
    src = source_by_path.get(path)
    assert src, path
    excerpt = '\n'.join(src.splitlines()[:85])
    display(Markdown(f'### `{path}`\n```python\n{excerpt}\n```'))

### `docs/cubit_mesh_export/hex_sphere_highorder/hex_sphere_curved_ngsolve.py`
```python
# -*- coding: utf-8 -*-
"""High-order CURVED HEX mesh in NGSolve -- the cubit-mesh-export capability.

Coreform Cubit meshes a sphere with HEXES ('volume scheme sphere') and exports a
high-order Netgen .vol (order 1/2/3) via the plugin command

    block 1 add hex all
    export netgen "hexsph_oN.vol" order N overwrite

NGSolve loads the .vol directly and integrates the volume; as the element order rises
the curved hex boundary captures the sphere and the volume converges to 4/3 pi r^3:

    order 1 (straight hex) :  ~ -23 %    (a coarse 56-hex faceted sphere)
    order 2 (curved hex)   :  ~ -0.2 %
    order 3 (curved hex)   :  ~ +0.1 %

That is high-order *hexahedral* FEM geometry inside NGSolve -- which the Netgen mesher
alone does not provide (it curves tetrahedra).  See packages/cubit-mesh-export.

GOTCHA (important): a high-order .vol already carries its curved mid-side nodes.  Load
it with plain ``Mesh(path)`` and DO NOT call ``mesh.Curve()`` -- ``mesh.Curve(p)``
re-curves from the underlying CAD geometry, which a loaded .vol does NOT have, so it
RESETS every element to straight-sided (the volume jumps back to the -23 % linear value).
(``mesh.Curve`` is the right call only when the mesh was built from an in-memory geometry,
e.g. a netgen.occ / SplineGeometry CAD object.)

Pure NGSolve -- no Cubit needed to RUN this (the three .vol files are committed beside
it).  Regenerate them with Cubit if you change the sphere.  Headless:

    python hex_sphere_curved_ngsolve.py
"""
import math
import os
from ngsolve import Mesh, Integrate, CoefficientFunction, VOL

R = 0.05
V_EXACT = 4.0 / 3.0 * math.pi * R ** 3
HERE = os.path.dirname(os.path.abspath(__file__))


def main():
    print(f"Curved-hex sphere in NGSolve  (R={R} m, V_exact={V_EXACT:.6e} m^3)")
    worst_hi = 0.0
    for order in (1, 2, 3):
        mesh = Mesh(os.path.join(HERE, f"hexsph_o{order}.vol"))   # load AS-IS (no .Curve)
        types = {}
        for el in mesh.Elements(VOL):
            k = str(el.type).replace("ET.", "")
            types[k] = types.get(k, 0) + 1
        V = Integrate(CoefficientFunction(1), mesh, VOL)
        err = (V - V_EXACT) / V_EXACT * 100.0
        print(f"  order {order}  {types}  V={V:.6e}  err={err:+.3f}%")
        if order >= 2:
            worst_hi = max(worst_hi, abs(err))
    assert worst_hi < 1.0, f"high-order hex volume err {worst_hi:.3f}% should be < 1%"
    print("PASS: curved high-order hex converges to the analytic sphere (order>=2 < 1%)")


if __name__ == "__main__":
    main()
```

### `validation_test/cubit_mesh_export/validation_vol_boundary_normal_vectors.py`
```python
"""Validation-class `.vol` boundary normal/vector-area rows.

Electromagnetic force extraction needs surface orientation before it needs a
large solver: Maxwell stress integrates ``T n dS``.  This example checks that
named Coreform/Cubit sidesets exported through Netgen `.vol` preserve enough
triangle orientation information to recover each boundary's vector area and
planar unit normal.

Run:

    python validation_test/cubit_mesh_export/validation_vol_boundary_normal_vectors.py
    python validation_test/cubit_mesh_export/validation_vol_boundary_normal_vectors.py --vol C:\\temp\\box.vol
"""

from __future__ import annotations

import argparse
import json
import math
import sys
from pathlib import Path


ROOT = Path(__file__).resolve().parents[2]
SRC = ROOT / "packages" / "radia-mcp" / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from radia_mcp.radia_ngsolve.netgen_vol import parse_netgen_tri_tet_vol, read_netgen_tri_tet_vol  # noqa: E402


OUT_JSON = Path(__file__).with_name("validation_vol_boundary_normal_vectors_summary.json")

BOX_SIX_BOUNDARY_VOL = """\
mesh3d
dimension
3
geomtype
0
facedescriptors
6
1 1 0 1 1
2 1 0 1 1
3 1 0 1 1
4 1 0 1 1
5 1 0 1 1
6 1 0 1 1
surfaceelements
12
5 5 1 0 3 1 3 2
5 5 1 0 3 1 4 3
6 6 1 0 3 5 6 7
6 6 1 0 3 5 7 8
3 3 1 0 3 1 2 6
3 3 1 0 3 1 6 5
4 4 1 0 3 4 7 3
4 4 1 0 3 4 8 7
1 1 1 0 3 1 5 8
1 1 1 0 3 1 8 4
2 2 1 0 3 2 3 7
2 2 1 0 3 2 7 6
volumeelements
12
1 4 1 3 2 9
1 4 1 4 3 9
1 4 5 6 7 9
1 4 5 7 8 9
1 4 1 2 6 9
1 4 1 6 5 9
1 4 4 7 3 9
1 4 4 8 7 9
1 4 1 5 8 9
1 4 1 8 4 9
1 4 2 3 7 9
1 4 2 7 6 9
points
9
0 0 0
2 0 0
2 3 0
0 3 0
0 0 5
2 0 5
2 3 5
0 3 5
```

## Placement Policy

For this presentation cycle, `docs/cubit_mesh_export` remains protected because MCP's cross-validation registry advertises `validation_test/cubit_mesh_export/validation_vol_*_summary.json`, and other examples still import the committed high-order hex sphere meshes. The docs layer is now the human-facing result-saved view; any future cleanup should first move runnable validation scripts to `validation_test/cubit_mesh_export` and update the registry in the same change.